# 04 Feature engineering: stats + FFT
Stats: mean/std/min/max/slope; FFT energy bands. Output: data/processed/features_tabular.parquet (N,F)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
path = DATA_PROCESSED / "timeseries_resampled.npz"
if not path.exists():
    raise FileNotFoundError("Run 03 first")
arr = np.load(path)["data"]
N, T, V = arr.shape
rows = []
for i in range(N):
    row = {"sample_id": i}
    for v in range(V):
        x = arr[i, :, v]
        row[f"v{v}_mean"] = float(np.mean(x))
        row[f"v{v}_std"] = float(np.std(x) or 0)
        row[f"v{v}_min"] = float(np.min(x))
        row[f"v{v}_max"] = float(np.max(x))
        row[f"v{v}_slope"] = float(np.polyfit(np.arange(T), x, 1)[0])
        fft_mag = np.abs(np.fft.fft(x))
        row[f"v{v}_fft_energy"] = float(np.sum(fft_mag[: T // 2]))
    rows.append(row)
feat = pd.DataFrame(rows)
feat.to_parquet(DATA_PROCESSED / "features_tabular.parquet", index=False)
print(feat.head())